# 📘 Week 4 · Day 28: 고급 기법 — NLP, 이미지, Pseudo-Labeling

## 🎯 학습 목표
- **텍스트**: TF-IDF부터 Transformer(BERT)까지
- **이미지**: Augmentation 심화 + TTA
- **Pseudo-Labeling**: 상위권 단골 기법
- **Adversarial Training / MixUp / CutMix** 개념
- 데이터 종류별 **Top-3 전략**

> ⚠️ 이 노트북은 **개념 위주** — 실제 대회 때 필요한 것만 골라서 적용합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

---

## 1. NLP 대회 전략

### 1.1 TF-IDF — 단단한 베이스라인

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

docs = [
    "This product is amazing. I love it.",
    "Terrible, would not recommend to anyone.",
    "Good value for the money.",
    "Worst purchase ever made.",
    "Highly recommended, fast shipping.",
    "Broken on arrival, very disappointing.",
    "Great quality and fast delivery!",
    "Not worth the price, poor material.",
]
y = [1, 0, 1, 0, 1, 0, 1, 0]

# TF-IDF 파라미터 (실전 기본값)
vec = TfidfVectorizer(
    ngram_range=(1, 2),    # uni + bi gram
    min_df=1,
    max_df=0.95,
    max_features=5000,
    sublinear_tf=True,     # log(tf) 변환
    strip_accents='unicode'
)
X_tfidf = vec.fit_transform(docs)
print("TF-IDF shape:", X_tfidf.shape)
print("Vocab sample:", list(vec.vocabulary_.keys())[:10])

clf = LogisticRegression(max_iter=1000)
scores = cross_val_score(clf, X_tfidf, y, cv=3, scoring='accuracy')
print(f"\nCV acc: {scores.mean():.3f}")

### 1.2 TF-IDF 응용 팁

| 트릭 | 효과 |
|------|------|
| `ngram_range=(1,3)` | 표현력 ↑, 메모리 ↑ |
| `char_wb` analyzer | 오타·typo에 강함 |
| `sublinear_tf=True` | 긴 문서 효과 완화 |
| SVD로 차원축소 | 희소→밀집 → GBM에 넣기 |
| word + char TF-IDF 둘 다 concat | 항상 성능 ↑ |

### 1.3 char-level TF-IDF (오타/명사구에 강력)

In [ ]:
vec_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=1)
X_char = vec_char.fit_transform(docs)
print("char-level TF-IDF shape:", X_char.shape)

# word + char 합치기
from scipy.sparse import hstack
X_combined = hstack([X_tfidf, X_char])
print("combined shape:", X_combined.shape)

### 1.4 BERT / HuggingFace Transformers (SOTA)

```python
from transformers import AutoTokenizer, AutoModel
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model     = AutoModel.from_pretrained('bert-base-uncased')

# 토크나이즈
enc = tokenizer(docs, padding=True, truncation=True, return_tensors='pt')
with torch.no_grad():
    out = model(**enc)
cls_emb = out.last_hidden_state[:, 0, :]   # [CLS] 벡터
# → LightGBM의 입력으로 사용하거나 fine-tuning
```

### 1.5 NLP 대회 표준 파이프라인
1. **Baseline**: TF-IDF + LogReg → CV 파악
2. **Level up**: TF-IDF word + char → LightGBM / linear SVC
3. **SOTA**: BERT fine-tuning (대회 도메인 모델 우선, 예: `deberta-v3-large`)
4. **앙상블**: 다른 seed / 다른 모델 (RoBERTa, DeBERTa, XLM-R)
5. **Post-processing**: confidence threshold 조정, rank averaging

---

## 2. 이미지 대회 전략

### 2.1 Augmentation 카탈로그

In [ ]:
# albumentations - Kaggle 이미지 대회 표준 라이브러리
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2

    train_aug = A.Compose([
        A.RandomResizedCrop(224, 224, scale=(0.8, 1.0)),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.HueSaturationValue(p=0.3),
        A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),  # CutOut
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    print("Augmentation pipeline OK")
except ImportError:
    print("(albumentations 미설치 - pip install albumentations)")

### 2.2 MixUp / CutMix — 정규화 끝판왕

In [ ]:
import torch

def mixup(x, y, alpha=0.4):
    '''x: (B, C, H, W), y: (B,) 클래스 인덱스'''
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0))
    x_mix = lam * x + (1 - lam) * x[idx]
    return x_mix, y, y[idx], lam

def cutmix(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.shape
    idx = torch.randperm(B)

    cut_rat = np.sqrt(1 - lam)
    cut_w = int(W * cut_rat); cut_h = int(H * cut_rat)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1 = max(0, cx - cut_w // 2); x2 = min(W, cx + cut_w // 2)
    y1 = max(0, cy - cut_h // 2); y2 = min(H, cy + cut_h // 2)

    x_out = x.clone()
    x_out[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - ((x2-x1) * (y2-y1) / (W * H))
    return x_out, y, y[idx], lam

# 학습 루프에서
# x_mix, y_a, y_b, lam = mixup(xb, yb)
# loss = lam * crit(model(x_mix), y_a) + (1-lam) * crit(model(x_mix), y_b)
print("MixUp/CutMix 함수 정의 완료")

### 2.3 TTA (Test Time Augmentation)

예측 시점에도 augment → 여러 버전 평균 → **무료 성능 향상**.

In [ ]:
def tta_predict(model, x, n_augs=5):
    '''
    model: 학습된 모델
    x: 원본 이미지 배치 (B, C, H, W)
    '''
    preds = [model(x)]                          # 원본
    preds.append(model(torch.flip(x, [3])))     # 수평 flip
    preds.append(model(torch.flip(x, [2])))     # 수직 flip
    # 90도 회전 등 추가 가능
    return torch.stack(preds).mean(0)

print("TTA helper 정의 완료")

### 2.4 이미지 대회 표준 파이프라인
1. **Backbone 선택**: EfficientNet / ConvNeXt / ViT (timm 라이브러리 활용)
2. **이미지 크기**: 작게 시작 → 크게 fine-tune (progressive resizing)
3. **Augmentation**: albumentations (Cutout + Mixup + CutMix)
4. **Pretrained weights**: ImageNet 또는 도메인 pretraining
5. **Loss**: CrossEntropy → LabelSmoothing → FocalLoss (불균형)
6. **Scheduler**: CosineAnnealing + warmup
7. **TTA**: 제출 시 필수

---

## 3. Pseudo-Labeling — 상위권의 비밀무기

### 아이디어
```
1. 모델을 train으로 학습
2. test에 예측 → 확률 높은(>0.95) 샘플의 예측을 "pseudo label"로 취급
3. train + pseudo-labeled test 으로 재학습
4. 예측 성능 ↑
```

### ⚠️ 주의
- **confidence threshold** 중요 (너무 낮으면 오류 전파)
- **원본 라벨이 맞아야** (train이 너무 작으면 편향)
- 보통 1~2회만 반복

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=3000, n_features=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Step 1: baseline
model = lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1, n_jobs=-1)
model.fit(X_tr, y_tr)
base_auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
print(f"Baseline AUC: {base_auc:.4f}")

# Step 2: test에 예측
test_proba = model.predict_proba(X_te)[:, 1]

# Step 3: 확률이 극단적인 샘플만 선택
hi = test_proba > 0.95
lo = test_proba < 0.05
confident_mask = hi | lo
print(f"Confident test samples: {confident_mask.sum()} / {len(test_proba)}")

# pseudo label 생성
pseudo_X = X_te[confident_mask]
pseudo_y = (test_proba[confident_mask] > 0.5).astype(int)

# Step 4: 재학습
X_pseudo = np.vstack([X_tr, pseudo_X])
y_pseudo = np.hstack([y_tr, pseudo_y])

model2 = lgb.LGBMClassifier(n_estimators=200, random_state=42, verbose=-1, n_jobs=-1)
model2.fit(X_pseudo, y_pseudo)
new_auc = roc_auc_score(y_te, model2.predict_proba(X_te)[:, 1])
print(f"After pseudo-labeling AUC: {new_auc:.4f}")

### 3.1 Pseudo-Labeling 변형

| 방법 | 설명 |
|------|------|
| **Hard** | 예측 확률 > threshold → 0/1 라벨 |
| **Soft** | 확률 자체를 라벨로 (regression 처럼) |
| **Self-distillation** | soft + temperature scaling |
| **Iterative** | 여러 번 반복 (2~3회) |

---

## 4. Adversarial Training — 분포 shift에 강해지기

입력에 **작은 노이즈** 를 더한 adversarial 예시로 학습하면 **robust**해짐.

### 개념 간단 구현

In [ ]:
import torch
import torch.nn as nn

# 모델에 adversarial noise 추가 (FGSM)
def fgsm_attack(x, y, model, loss_fn, eps=0.01):
    x_adv = x.clone().detach().requires_grad_(True)
    loss = loss_fn(model(x_adv), y)
    loss.backward()
    x_adv = x_adv + eps * x_adv.grad.sign()
    return x_adv.detach()

# 학습 루프에서
# x_adv = fgsm_attack(xb, yb, model, criterion)
# loss = 0.5 * criterion(model(xb), yb) + 0.5 * criterion(model(x_adv), yb)
print("FGSM helper 준비 완료")

---

## 5. 기타 Kaggle 상위권 꿀팁

### 5.1 타깃 변환 (Regression)
- `np.log1p` — 왜도 큰 타깃
- Box-Cox — 음수 없을 때
- Yeo-Johnson — 음수 포함 가능
- Quantile transform — 타깃 분포를 정규분포로

### 5.2 Post-processing
- Classification: **threshold 튜닝** (0.5 고정 금지!)
- Regression: **예측값 clip** (train 범위 밖으로 나가면 위험)
- Ranking: **isotonic regression**으로 확률 calibration

### 5.3 제출물 다양성 확보
- public LB에 맞춰 2~3개 다른 접근 제출
- private shake-up 대비해 **CV 신뢰 제출** 1개 + **LB 상위 제출** 1개
- 마지막 선택(final submission)은 CV 기준 **가장 robust한 것**을 우선

---

## 6. 데이터 유형별 Top-3 전략 요약

| 유형 | 1순위 | 2순위 | 3순위 |
|------|-------|-------|-------|
| Tabular | LightGBM | CatBoost | XGBoost + NN (스태킹) |
| Image | EfficientNet/ConvNeXt + TTA | ViT | 2~3 fold + MixUp |
| NLP | DeBERTa-v3 fine-tune | RoBERTa | TF-IDF + LGBM baseline |
| Time-series | LGBM + lag feat | N-BEATS/TFT | Prophet / SARIMA |
| Signal/Audio | 1D CNN | Mel-spectrogram + 2D CNN | Wav2Vec2 |

---

## 📝 Day 28 체크리스트
- [ ] TF-IDF + LogReg 베이스라인 가능
- [ ] MixUp / CutMix 이해
- [ ] TTA 의미 이해
- [ ] Pseudo-labeling 개념 이해 + 실행
- [ ] 데이터 유형별 1순위 모델 매핑

다음 노트북에서 **Kaggle Notebook 환경 활용, 제출 체크리스트, reproducibility** 를 다룹니다.